# Детекция ботов в диалогах. Мета-стекинг и калибровка

Постановка задачи: построение двухуровневого ансамбля (Level-2 Meta-Stacker) на базе предсказаний разнородных моделей 1-го уровня (градиентные бустинги над таблично-семантическими признаками и нейросетевой классификатор RuBERT).

Цели этапа:
- Извлечение честных OOF и тестовых предсказаний моделей 1-го уровня из ClickHouse DWH (`model_predictions`).
- Преобразование вероятностей в пространство логитов (Log-Odds) для устранения нелинейности сигмоиды.
- Оптимизация регуляризации мета-модели через `TwoLevelStackingTrainer` и `StackingSearchConfig` в Optuna.
- Кросс-валидация мета-стекера (5-Fold GroupKFold) с парной постобработкой диалогов (`PairwisePostprocessor`).
- Сохранение предсказаний в ClickHouse DWH и регистрация продакшн-чемпиона в MLflow Model Registry (`bot_detector`).

In [1]:
import warnings

import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import polars as pl
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score

from project import Reproducibility
from project.core.config import config
from project.core.utils import LogitTransformer
from project.infrastructure.clickhouse import FeatureDWH
from project.models.classical import (
    ClassicalTrainerConfig,
    StackingSearchConfig,
    TwoLevelStackingClassifier,
    TwoLevelStackingTrainer,
)

warnings.filterwarnings("ignore", category=UserWarning)

2026/11/10 00:06:42 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /home/alex/cu/prod-ml/a.gorobchenko/project/.venv/lib/python3.13/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


In [2]:
Reproducibility.seed(42)

## Загрузка данных и разметки из ClickHouse DWH

Выгружаем сырые диалоги, целевые метки классов и идентификаторы диалогов для корректного разбиения по фолдам (GroupKFold).

In [3]:
dwh = FeatureDWH()
train_dialogs = dwh.raw.load_dialogues_dict("train")
test_dialogs = dwh.raw.load_dialogues_dict("test")
ytrain = dwh.raw.load_labels("train")
ytest = dwh.raw.load_labels("test")

y_train = ytrain["is_bot"].to_numpy().astype(int)
groups = ytrain["dialog_id"].to_numpy()

pl.DataFrame(
    {
        "split": ["train", "test"],
        "dialogues_count": [len(train_dialogs), len(test_dialogs)],
        "participants_count": [len(ytrain), len(ytest)],
        "bot_share": [round(float(np.mean(y_train)), 4), None],
    }
)

split,dialogues_count,participants_count,bot_share
str,i64,i64,f64
"""train""",1516,3032,0.408
"""test""",379,758,null


## Загрузка OOF и тестовых предсказаний моделей Level-1

Извлекаем сохраненные в ClickHouse DWH предсказания моделей первого уровня:
- `catboost_rubert_hybrid` (градиентный бустинг CatBoost над отобранными табличными и семантическими признаками)
- `lightgbm_rubert_hybrid` (градиентный бустинг LightGBM над отобранными признаками)
- `xgboost_rubert_hybrid` (градиентный бустинг XGBoost над отобранными признаками)
- `rubert_tiny2_semantic_classifier` (нейросетевой классификатор RuBERT-tiny2)

In [4]:
client = mlflow.tracking.MlflowClient(config.mlflow_tracking_uri)
exp = client.get_experiment_by_name("bot-detection")
if not exp:
    raise RuntimeError("Эксперимент 'bot-detection' не найден в MLflow!")


def get_latest_run_id(run_name: str) -> str:
    runs = client.search_runs(
        exp.experiment_id,
        filter_string=f"attributes.run_name = '{run_name}' and attributes.status = 'FINISHED'",
        order_by=["attributes.start_time DESC"],
    )
    if not runs:
        runs = client.search_runs(
            exp.experiment_id,
            filter_string=f"tags.mlflow.runName = '{run_name}' and attributes.status = 'FINISHED'",
            order_by=["attributes.start_time DESC"],
        )
    if not runs:
        raise RuntimeError(f"Не найден завершенный MLflow-ран '{run_name}'!")
    return runs[0].info.run_id


level1_models = [
    "catboost_rubert_hybrid",
    "lightgbm_rubert_hybrid",
    "xgboost_rubert_hybrid",
    "rubert_tiny2_semantic_classifier",
]

oof_preds: dict[str, np.ndarray] = {}
test_preds: dict[str, np.ndarray] = {}
level1_runs: dict[str, str] = {}

for m_name in level1_models:
    rid = get_latest_run_id(m_name)
    level1_runs[m_name] = rid
    tr_df = dwh.predictions.load_predictions(run_id=rid, dataset_split="train_oof")
    te_df = dwh.predictions.load_predictions(run_id=rid, dataset_split="test")

    tr_aligned = ytrain.select(["dialog_id", "participant_index"]).join(
        tr_df.select(["dialog_id", "participant_index", "prob_bot"]),
        on=["dialog_id", "participant_index"],
        how="left",
    )
    te_aligned = ytest.select(["dialog_id", "participant_index"]).join(
        te_df.select(["dialog_id", "participant_index", "prob_bot"]),
        on=["dialog_id", "participant_index"],
        how="left",
    )
    oof_preds[m_name] = tr_aligned["prob_bot"].to_numpy()
    test_preds[m_name] = te_aligned["prob_bot"].to_numpy()

pl.DataFrame(
    [
        {
            "model": m_name,
            "run_id": level1_runs[m_name][:8],
            "oof_logloss": round(float(log_loss(y_train, oof_preds[m_name])), 5),
            "oof_roc_auc": round(float(roc_auc_score(y_train, oof_preds[m_name])), 5),
            "oof_accuracy": round(float(accuracy_score(y_train, (oof_preds[m_name] >= 0.5).astype(int))), 5),
        }
        for m_name in level1_models
    ]
)

model,run_id,oof_logloss,oof_roc_auc,oof_accuracy
str,str,f64,f64,f64
"""catboost_rubert_hybrid""","""96923886""",0.16211,0.98292,0.93734
"""lightgbm_rubert_hybrid""","""e0a9447f""",0.16857,0.98164,0.93272
"""xgboost_rubert_hybrid""","""59c90015""",0.16395,0.98274,0.93371
"""rubert_tiny2_semantic_classifi…","""59637170""",0.69075,0.52607,0.56563


## Формирование мета-признакового пространства

Для каждого участника переводим вероятности моделей первого уровня в логиты (Log-Odds):
$$\text{logit}(p) = \ln \left( \frac{p}{1 - p} \right)$$
Логит-трансформация устраняет нелинейность сигмоиды и сжатие вблизи границ 0 и 1, позволяя линейному классификатору эффективно комбинировать сигналы.

In [5]:
meta_feature_names = [f"logit_{m}" for m in level1_models]

Z_tr = np.column_stack([LogitTransformer.transform(oof_preds[m]) for m in level1_models])
Z_te = np.column_stack([LogitTransformer.transform(test_preds[m]) for m in level1_models])

pl.DataFrame(
    {
        "meta_feature": meta_feature_names,
        "mean": [round(float(np.mean(Z_tr[:, i])), 4) for i in range(len(meta_feature_names))],
        "std": [round(float(np.std(Z_tr[:, i])), 4) for i in range(len(meta_feature_names))],
        "min": [round(float(np.min(Z_tr[:, i])), 4) for i in range(len(meta_feature_names))],
        "max": [round(float(np.max(Z_tr[:, i])), 4) for i in range(len(meta_feature_names))],
    }
)

meta_feature,mean,std,min,max
str,f64,f64,f64,f64
"""logit_catboost_rubert_hybrid""",-0.8895,4.5344,-9.2172,9.9165
"""logit_lightgbm_rubert_hybrid""",-0.8867,4.414,-8.8076,8.75
"""logit_xgboost_rubert_hybrid""",-0.8742,4.3157,-8.2098,8.2439
"""logit_rubert_tiny2_semantic_cl…",-0.3888,0.4372,-2.6745,1.0614


## Оптимизация и обучение мета-стекера через TwoLevelStackingTrainer

Используем стандартный фасад `TwoLevelStackingTrainer`:
- Оптимизирует силу регуляризации $C$ и параметр `fit_intercept` через `StackingSearchConfig` в Optuna.
- Выполняет 5-Fold GroupKFold валидацию с парной диалоговой калибровкой `PairwisePostprocessor`.
- Логирует метрики, параметры, дерево ансамбля и графики в MLflow Tracking.
- Сохраняет честные OOF и Test предсказания в ClickHouse DWH (`model_predictions`).
- Регистрирует модель в MLflow Model Registry как `bot_detector`.

In [6]:
trainer_config = ClassicalTrainerConfig(
    experiment_name="bot-detection",
    run_name="logistic_meta_stacker",
    description=(
        "Двухуровневый мета-стекер TwoLevelStackingClassifier: логистическая регрессия "
        "над логитами CatBoost, LightGBM, XGBoost и RuBERT с парной калибровкой диалогов."
    ),
    n_trials=300,
    patience=50,
    n_splits=5,
    random_state=42,
    register_as="bot_detector",
    dataset_name="train_meta_level2",
)

search_config = StackingSearchConfig(c_min=0.001, c_max=10.0)

stacking_trainer = TwoLevelStackingTrainer(
    config=trainer_config,
    search_config=search_config,
    level1_runs=level1_runs,
    use_logits=True,
)

# Подбор гиперпараметров через Optuna
tuning_result = stacking_trainer.tune(
    X=Z_tr,
    y=y_train,
    groups=groups,
    feature_names=meta_feature_names,
)
print(f"Лучшие параметры мета-модели: {tuning_result.best_params}")
print(f"Лучший CV LogLoss: {tuning_result.best_score:.5f}")

# Финальное обучение, калибровка, логирование и регистрация
training_result = stacking_trainer.fit(
    X_train=Z_tr,
    y_train=y_train,
    groups=groups,
    X_test=Z_te,
    dialog_ids_train=ytrain["dialog_id"].to_numpy(),
    participant_indices_train=ytrain["participant_index"].to_numpy(),
    dialog_ids_test=ytest["dialog_id"].to_numpy(),
    participant_indices_test=ytest["participant_index"].to_numpy(),
    test_ids=ytest["ID"].to_numpy(),
    feature_names=meta_feature_names,
)

[I 2026-11-10 00:06:49,428] A new study created in memory with name: no-name-397ab58f-ba1b-407a-8fd3-b7ffa2dc2d87


[I 2026-11-10 00:06:49,450] Trial 0 finished with value: 0.1609136611182492 and parameters: {'C': 0.03148911647956861, 'fit_intercept': True}. Best is trial 0 with value: 0.1609136611182492.


[I 2026-11-10 00:06:49,581] Trial 1 finished with value: 0.16095047360585713 and parameters: {'C': 0.24810409748678125, 'fit_intercept': True}. Best is trial 0 with value: 0.1609136611182492.


[I 2026-11-10 00:06:49,703] Trial 2 finished with value: 0.16683796084263483 and parameters: {'C': 0.0017073967431528124, 'fit_intercept': True}. Best is trial 0 with value: 0.1609136611182492.


[I 2026-11-10 00:06:49,832] Trial 3 finished with value: 0.1610761431790521 and parameters: {'C': 0.679657809075816, 'fit_intercept': False}. Best is trial 0 with value: 0.1609136611182492.


[I 2026-11-10 00:06:49,982] Trial 4 finished with value: 0.16112726232796729 and parameters: {'C': 2.1368329072358767, 'fit_intercept': True}. Best is trial 0 with value: 0.1609136611182492.


... (118 outputs left out)

2026/11/10 00:07:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Registered model 'bot_detector' already exists. Creating a new version of this model...
2026/11/10 00:07:14 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: bot_detector, version 4


Created version '4' of model 'bot_detector'.


🏃 View run logistic_meta_stacker at: http://127.0.0.1:5000/#/experiments/7/runs/67d862f75ae24c5c81a26dcc0eb92034
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7


## Итоги обучения и валидации мета-стекера

Проверяем ключевые метрики качества классификатора и архитектуру зарегистрированного ансамбля.

In [7]:
pl.DataFrame(
    {
        "metric": list(training_result.metrics.keys()),
        "value": [round(float(v), 5) for v in training_result.metrics.values()],
    }
)

metric,value
str,f64
"""val_logloss""",0.15967
"""val_logloss_raw""",0.16084
"""val_roc_auc""",0.98376
"""val_accuracy""",0.93734
"""val_f1""",0.9211
